# Simulator method metrics (breast-level)

Per-method, per-site **and** pooled breast-level metrics from the NVFLARE-simulator prediction CSVs.

- **Unit:** breast (mean prob / max label per (exam, laterality)) via `fl_utils.breast_aggregate`.
- **Sources by method type:**
  - *Shared-model* (FedAvg / FedProx / FedBN): per-round `incoming_global` CSVs (received-aggregate eval each round; for **FedBN** = global backbone + that site's local BN).
  - *Personalized* (Ditto / module-wise): the run saves a per-round personal-model checkpoint (`<site>_gmic_model_round_*.pth`) but no per-round preds. Run **`dump_ditto_perround_preds.py`** first to dump per-round preds (tag `*_perround`) from those checkpoints — then Ditto is treated **exactly like the shared methods** (`trajectory=True`), fully apples-to-apples.
- **Round selection (all trajectory methods):** (A) each site's own best-val-AUC round; (B) one common round maximizing **pooled** val AUC.
- **Operating point:** Youden's J on validation -> test (per-site for site rows; pooled-val for the Pooled row).
- **CIs:** AUC -> DeLong **and** bootstrap; sensitivity/specificity -> bootstrap.

Edit the **Config** cell then **Run All**. Missing methods self-skip with a note. Writes two CSVs + a markdown table to `OUT_DIR`.

In [ ]:
# ---- Config: edit paths / methods here, then Run All ----
import os, re, glob, csv, json, sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import norm
from IPython.display import display

# Reuse the codebase's EXACT breast aggregation (mean prob, max label per (exam, laterality)).
# Point at any sim job's app/custom (identical across jobs); adjust to your checkout.
FL_UTILS_DIR = "/raid/home/lsollis/GMIC/GMIC/gmic_job_fedbn_sim/app/custom"

CLIENTS = ["UHCC", "HPU", "RSNA-GCP"]

# Per method: dir template, prediction-CSV tag, and whether it has a per-round trajectory.
#   trajectory=True -> per-round preds available -> per-site best-val + pooled-best-round selection.
#   Shared methods (FedAvg/FedProx/FedBN) dump tag=incoming_global every round natively.
#   Personalized methods (Ditto family): FIRST run dump_ditto_perround_preds.py to dump the
#   per-round PERSONAL-model preds (tag=*_perround) from the saved <site>_gmic_model_round_*.pth;
#   they are then treated exactly like the shared methods (trajectory=True), fully apples-to-apples.
BASE = "/raid/home/lsollis/GMIC/GMIC/sim"
METHODS = {
    "FedProx (mu=0.1)":         {"dir": BASE + "/fedprox_mu0.1/{site}",          "tag": "incoming_global",           "trajectory": True},
    "FedBN":                    {"dir": BASE + "/fedbn/{site}",                  "tag": "incoming_global",           "trajectory": True},
    "FedAvg":                   {"dir": BASE + "/fedavg/fedavg/{site}",          "tag": "incoming_global",           "trajectory": True},
    "Ditto (lambda=0.05)":      {"dir": BASE + "/ditto_l0.05/{site}",            "tag": "ditto_perround",            "trajectory": True},
    # Uncomment after running dump_ditto_perround_preds.py for the module-wise job:
    # "Ditto-MW (g0.01/l0.5/f0)": {"dir": BASE + "/ditto_mw_g0.01_l0.5_f0/{site}", "tag": "ditto_modulewise_perround", "trajectory": True},
}
N_BOOT  = 2000
SEED    = 0
CI      = 0.95
OUT_DIR = "/raid/home/lsollis/GMIC/GMIC"

In [ ]:
# ---- Helpers: aggregation, AUC + DeLong CI, Youden threshold, point metrics, bootstrap ----
sys.path.insert(0, FL_UTILS_DIR)
from fl_utils import breast_aggregate   # canonical view -> breast aggregation

def safe_auc(labels, scores):
    labels = np.asarray(labels)
    return float(roc_auc_score(labels, scores)) if len(np.unique(labels)) > 1 else float("nan")

def delong_ci(labels, scores, alpha=CI):
    """AUC + DeLong CI for a single classifier (placement-value form; exact, O(m*n))."""
    labels = np.asarray(labels).astype(int); scores = np.asarray(scores, float)
    pos = scores[labels == 1]; neg = scores[labels == 0]
    m, n = len(pos), len(neg)
    if m == 0 or n == 0:
        return float("nan"), float("nan"), float("nan")
    cmp = (pos[:, None] > neg[None, :]).astype(float) + 0.5 * (pos[:, None] == neg[None, :])
    auc = float(cmp.mean())
    if m < 2 or n < 2:
        return auc, float("nan"), float("nan")
    V10 = cmp.mean(axis=1); V01 = cmp.mean(axis=0)
    se = float(np.sqrt(V10.var(ddof=1) / m + V01.var(ddof=1) / n))
    z = float(norm.ppf(1 - (1 - alpha) / 2))
    return auc, max(0.0, auc - z * se), min(1.0, auc + z * se)

def youden_threshold(labels, scores):
    fpr, tpr, thr = roc_curve(labels, scores)
    return float(thr[int(np.argmax(tpr - fpr))])

def point_metrics(labels, scores, thr):
    labels = np.asarray(labels).astype(int); scores = np.asarray(scores, float)
    pred = (scores >= thr).astype(int)
    tp = int(((pred == 1) & (labels == 1)).sum()); tn = int(((pred == 0) & (labels == 0)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum()); fn = int(((pred == 0) & (labels == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    return sens, spec

def bootstrap_ci(labels, scores, thr, n_boot=N_BOOT, seed=SEED, alpha=CI):
    labels = np.asarray(labels).astype(int); scores = np.asarray(scores, float)
    rng = np.random.default_rng(seed); idx = np.arange(len(labels))
    aucs, senss, specs = [], [], []
    for _ in range(n_boot):
        b = rng.choice(idx, len(idx), replace=True)
        yb, sb = labels[b], scores[b]
        if len(np.unique(yb)) < 2:
            continue
        aucs.append(roc_auc_score(yb, sb))
        s, p = point_metrics(yb, sb, thr); senss.append(s); specs.append(p)
    def q(a):
        if not a:
            return (float("nan"), float("nan"))
        return (float(np.nanpercentile(a, 100 * (1 - alpha) / 2)),
                float(np.nanpercentile(a, 100 * (1 + alpha) / 2)))
    return q(aucs), q(senss), q(specs)

def load_breast(method_tmpl, site, split, tag):
    """round -> (breast_probs, breast_labels) from the per-view prediction CSVs for this tag."""
    d = method_tmpl.format(site=site)
    out = {}
    for path in glob.glob(os.path.join(d, f"{site}_predictions_{tag}_round*_{split}.csv")):
        mo = re.search(r"_round(\d+)_", os.path.basename(path))
        if not mo:
            continue
        with open(path, newline="") as f:
            rows = list(csv.DictReader(f))
        if not rows:
            continue
        bp, by = breast_aggregate([r["exam_id"] for r in rows], [r["view"] for r in rows],
                                  [float(r["prob_malignant"]) for r in rows], [int(r["label"]) for r in rows])
        out[int(mo.group(1))] = (np.asarray(bp, float), np.asarray(by, int))
    return out

In [ ]:
# ---- Build the two tables: (A) per-site best-val round, (B) pooled best round ----
def _row(method, site, rnd, by_test, bp_test, thr):
    auc, dlo, dhi = delong_ci(by_test, bp_test)
    (alo, ahi), (slo, shi), (plo, phi) = bootstrap_ci(by_test, bp_test, thr)
    sens, spec = point_metrics(by_test, bp_test, thr)
    ci = lambda lo, hi: f"[{lo:.3f}, {hi:.3f}]"
    return {"method": method, "site": site, "round": int(rnd),
            "n_breasts": int(len(by_test)), "prevalence": round(float(np.mean(by_test)), 4),
            "AUC": round(auc, 4), "AUC_DeLong95": ci(dlo, dhi), "AUC_boot95": ci(alo, ahi),
            "threshold": round(float(thr), 4),
            "sensitivity": round(sens, 4), "sens_boot95": ci(slo, shi),
            "specificity": round(spec, 4), "spec_boot95": ci(plo, phi)}

rows_persite, rows_pooled = [], []
for name, cfg in METHODS.items():
    tmpl, tag, traj = cfg["dir"], cfg["tag"], cfg.get("trajectory", True)
    val  = {s: load_breast(tmpl, s, "val",  tag) for s in CLIENTS}
    test = {s: load_breast(tmpl, s, "test", tag) for s in CLIENTS}
    if any(not val[s] or not test[s] for s in CLIENTS):
        print(f"[skip] {name}: missing CSVs ->")
        for s in CLIENTS:
            print(f"    {s}: {len(val[s])} val / {len(test[s])} test rounds  in {tmpl.format(site=s)}")
        continue

    if traj:
        # (A) per-site best-val round: each site picks its own round by its val AUC
        for s in CLIENTS:
            r = max(val[s], key=lambda rr: safe_auc(val[s][rr][1], val[s][rr][0]))
            thr = youden_threshold(val[s][r][1], val[s][r][0])
            rows_persite.append(_row(name, s, r, test[s][r][1], test[s][r][0], thr))
        # (B) pooled best round: one round (present at all sites) maximizing POOLED val AUC
        common = sorted(set.intersection(*[set(val[s]) & set(test[s]) for s in CLIENTS]))
        rp = max(common, key=lambda r: safe_auc(np.concatenate([val[s][r][1] for s in CLIENTS]),
                                                np.concatenate([val[s][r][0] for s in CLIENTS])))
        for s in CLIENTS:
            thr = youden_threshold(val[s][rp][1], val[s][rp][0])
            rows_pooled.append(_row(name, s, rp, test[s][rp][1], test[s][rp][0], thr))
        by_v = np.concatenate([val[s][rp][1] for s in CLIENTS]); bp_v = np.concatenate([val[s][rp][0] for s in CLIENTS])
        by_t = np.concatenate([test[s][rp][1] for s in CLIENTS]); bp_t = np.concatenate([test[s][rp][0] for s in CLIENTS])
        rows_pooled.append(_row(name, "Pooled", rp, by_t, bp_t, youden_threshold(by_v, bp_v)))
    else:
        # Personalized (Ditto family): tag=*_bestval from dump_bestval_preds.py -> the BEST-VAL
        # personal model, one round (the best-val round). Best-val-selected, so apples-to-apples
        # with the shared methods; the same single-round result fills both tables.
        rstar = {s: max(val[s]) for s in CLIENTS}   # single (best-val) round per site
        for s in CLIENTS:
            r = rstar[s]
            thr = youden_threshold(val[s][r][1], val[s][r][0])
            row = _row(name, s, r, test[s][r][1], test[s][r][0], thr)
            rows_persite.append(row)
            rows_pooled.append(dict(row))
        by_v = np.concatenate([val[s][rstar[s]][1] for s in CLIENTS]); bp_v = np.concatenate([val[s][rstar[s]][0] for s in CLIENTS])
        by_t = np.concatenate([test[s][rstar[s]][1] for s in CLIENTS]); bp_t = np.concatenate([test[s][rstar[s]][0] for s in CLIENTS])
        rows_pooled.append(_row(name, "Pooled", rstar[CLIENTS[0]], by_t, bp_t, youden_threshold(by_v, bp_v)))

df_persite = pd.DataFrame(rows_persite)
df_pooled  = pd.DataFrame(rows_pooled)

In [ ]:
# ---- Display + export ----
print("=== (A) PER-SITE best-val-round selection (each site at its own best val round) ===")
display(df_persite)
print("\n=== (B) POOLED best-round selection (common round maximizing pooled val AUC) ===")
display(df_pooled)

os.makedirs(OUT_DIR, exist_ok=True)
df_persite.to_csv(os.path.join(OUT_DIR, "metrics_persite_bestval.csv"), index=False)
df_pooled.to_csv(os.path.join(OUT_DIR, "metrics_pooled_bestround.csv"), index=False)
try:
    md = ("# Simulator metrics (breast-level, Youden-on-val operating point)\n\n"
          "## (A) Per-site best-val-round\n\n" + df_persite.to_markdown(index=False) +
          "\n\n## (B) Pooled best-round\n\n" + df_pooled.to_markdown(index=False) + "\n")
    with open(os.path.join(OUT_DIR, "metrics_for_paper.md"), "w") as f:
        f.write(md)
    print("\nwrote metrics_for_paper.md (+ 2 CSVs) to", OUT_DIR)
except Exception as e:
    print("markdown export skipped (pip install tabulate for .md):", e)
    print("CSVs written to", OUT_DIR)